### Import 


In [4]:
import os
import json
import requests
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup

### Ceneo Web Scraper

1. provide URLof the product's opinion page

In [5]:
product_code = "39747497"
page = 1
url = f"https://www.ceneo.pl/{product_code}/opinie-{page}"
print(url)

https://www.ceneo.pl/39747497/opinie-1


2. send request to provided URL

In [6]:
response = requests.get(url)
print(response.status_code)

200


0. prepare selenium

In [10]:
path_to_driver = "D:\\selenium\\chromedriver-win64\\chromedriver.exe"
s = Service(path_to_driver)
driver = webdriver.Chrome(service=s)
driver.get(url)
driver.maximize_window()
driver.find_element(by="xpath", value ='//*[@id="js_cookie-consent-general"]/div/div[2]/button[1]').click()

3. fetch product name


In [4]:
page_dom = BeautifulSoup(response.text, "html.parser")
print(type(page_dom))

<class 'bs4.BeautifulSoup'>


In [5]:
Product_Name = page_dom.select_one('h1').get_text()
print(Product_Name)
print(type(Product_Name))

Gerlach Patelnia Prestige 28cm
<class 'str'>


In [6]:
Product_Name = page_dom.find('h1').get_text()
print(Product_Name)
print(type(Product_Name))

Gerlach Patelnia Prestige 28cm
<class 'str'>


4. fetch all opinions from the webpage

In [14]:
# more versitile compeared to find all

Opinions = page_dom.select("div.js_product-review:not(.user-post--highlight)")

print(type(Opinions))
print(len(Opinions))

<class 'bs4.element.ResultSet'>
10


5. Parse opinions to extract requiered data

In [8]:
all_opinions =[]
for opinion in Opinions:
    single_opinion = {
        'opinion_id': opinion.get("data-entry-id"),
        'author': opinion.select_one("span.user-post__author-name").get_text().strip(),
        'recomendations': opinion.select_one('span.user-post__author-recomendation > em').get_text().strip() if opinion.select_one('span.user-post__author-recomendation > em') else None,
        'score': opinion.select_one('span.user-post__score-count').get_text().strip(),
        'content': opinion.select_one('div.user-post__text').get_text().strip(),
        'pros': [p.get_text() for p in opinion.select('div.review-feature__item--positive')],
        'cons': [c.get_text() for c in opinion.select('div.review-feature__item--negative')],
        'helpfull': opinion.select_one('button.vote.yes > span').get_text().strip() if opinion.select_one('button.vote.yes > span') else None,
        'unhelpfull': opinion.select_one('button.vote.no > span').get_text().strip() if opinion.select_one('button.vote.no > span') else None,
        'publishing_date': opinion.select_one('span.user-post__published > time:nth-child(1)[datetime]').get('datetime').strip(),
        'purchese_date': opinion.select_one('span.user-post__published > time:nth-child(2)[datetime]').get('datetime').strip() if opinion.select_one('span.user-post__published > time:nth-child(2)[datetime]') else None,
    }
    all_opinions.append(single_opinion)
print(all_opinions)

[{'opinion_id': '12960738', 'author': 'Norbert', 'recomendations': 'Polecam', 'score': '4,5/5', 'content': 'Patelnia masywna, solidnie wykonana. Trzeba na początek trochę cierpliwości, aby przestawić się ze zwykłych teflonowych na cało-stalową, ponieważ trochę inaczej się na niej smaży oraz ją myje, ale po kilku razach wydaje się być lepsza od tych teflonowych. Mam nadzieję, że przetrwa długie lata i będzie solidniejsza od teflonowych, kolorowanych z zewnątrz patelni.', 'pros': ['funkcjonalność', 'wygląd', 'wytrzymałość'], 'cons': [], 'helpfull': None, 'unhelpfull': None, 'publishing_date': '2020-08-16 11:03:22', 'purchese_date': '2020-06-29 12:22:30'}, {'opinion_id': '6536885', 'author': 'Helena', 'recomendations': 'Polecam', 'score': '5/5', 'content': 'Bardzo polecam tym co cenią jakość. w kuchni rewelacja, mało smażymy aledo zapiekania-rewelacja, cała jest metalowa co spokojnie zdaje egzamin w piekarniku, mniej zmywania, polecam', 'pros': ['funkcjonalność', 'wygląd', 'wytrzymałość']

6. check if there is a next page with opinions

In [ ]:
driver.find_element (by="xpath", value = '//*[@id="reviews"]/div/div[6]/button[4]').click()
#IDK what this is

In [9]:
next = True if page_dom.select_one("button.pagination__next") else False
print(next)
if next: page += 1


True


8. save aquiered opinions

In [10]:
if not os.path.exists("./opinions"):
    os.mkdir("./opinions")

In [11]:
with open(f"./opinions/{product_code}.json", "w", encoding="UTF-8") as jf:
    json.dump(all_opinions, jf, indent=4, ensure_ascii=False)